In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
from sklearn.preprocessing import normalize
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from PIL import Image
from lucent.optvis import render, param, transform, objectives
import shelve
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import random

import matplotlib.pyplot as plt
import numpy as np
from lucent.modelzoo import inceptionv1
from PIL import Image
from torch.nn import functional as F
from lucent.optvis import param

from olt.act import InputOutputModelSnapshot
import json

from lucent.modelzoo import inceptionv1
from olt.tfms import transform
from olt.act import InputOutputModelSnapshot
from olt.show import show_single_channel_red_green_black as S
from olt.shards import raw_iter_shards, read_image_shard


base_report_dir = Path("mass-train-reports")
plt.style.use("dark_background")


device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


FLAT_IMAGE_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/hiccup-ide/olt/notebooks/this-and-prev/flat-images"
)


In [ ]:
from olt.show import show_single_channel_red_green_black as S
from sklearn.cluster import KMeans


def _receptive_block(i, ksize, stride, padding, input_size=None):
    """
    Returns [start, end) input indices (end=exclusive) that influence
    output position i of a conv layer.
    """
    start = i * stride - padding
    end = start + ksize

    if input_size is not None:
        start = max(start, 0)
        end = min(end, input_size)

    return start, end

def get_grid_of_channels_act_picker(model, layer_name, channel_shape):
    n_inp_channels, in_h, in_w = model.get_submodule(layer_name).weight.shape[1:]
    H, W = channel_shape
    def rsh(act):
        act = act.reshape(n_inp_channels, in_h, in_w)
        grid = np.zeros((in_h * H, in_w * W), dtype=act.dtype)
        for y in range(in_h):
            for x in range(in_w):
                cell = act[:, y, x].reshape(H, W)
                grid[y*H:(y+1)*H, x*W:(x+1)*W] = cell
        return grid
    return rsh

def vis_weight(model, layer_name, channel, ncomps=5):
    w = model.get_submodule(layer_name).weight[channel].detach().cpu()
    height, width = w.shape[-2:]
    chan_last_w = w.reshape(w.shape[0], -1).permute(1, 0).numpy()  # [pos, chan]

    _, axes = plt.subplots(1, 2, figsize=(15, 5))

    ins = [KMeans(c).fit(chan_last_w).inertia_ for c in range(1, height * width)]
    axes[0].plot(ins)

    # ideal comps is not 5 though
    km = KMeans(ncomps).fit(chan_last_w)
    axes[1].imshow(km.labels_.reshape(height, width), cmap="tab10")

    plt.show()


def show_for_row(model, row):
    pil = Image.open(FLAT_IMAGE_DIR / f"{row.input_image_key}.jpeg")
    timg = transform(pil)[None]
    act = InputOutputModelSnapshot.get_activations(timg, model, [row.layer_name])[row.layer_name]["input"]
    layer = model.get_submodule(row.layer_name)
    ksize, stride, padding = layer.kernel_size, layer.stride, layer.padding
    y0,y1 = _receptive_block(row.y_position, ksize[0], stride[0], padding[0])
    x0,x1 = _receptive_block(row.x_position, ksize[1], stride[1], padding[1])

    picker = get_grid_of_channels_act_picker(model, row.layer_name, (6,8))
    reshaped = picker(act[0, :, y0:y1, x0:x1].numpy())

    flat_w = layer.weight[row.channel].reshape(-1).detach().cpu().numpy()
    picked_w = picker(flat_w)
    S([act[0, :, y0:y1, x0:x1].reshape(30,40), reshaped, picked_w, reshaped*picked_w], (10,2), 4, viztype="local", suptitle=f"{row.imagenet_label}")
    plt.show()

    

In [ ]:
df = pd.read_csv("weight-banding/mixed5b_5x5_pre_relu_conv/12/report.csv")
df = df[df.cluster_label != -1]
df.head()

In [ ]:
rows = df[df.cluster_label == 15].iloc[:5]

for row in rows.itertuples():
    show_for_row(model, row)

In [ ]:
rows = df[(df.cluster_label == 28)].iloc[:10]

for row in rows.itertuples():
    show_for_row(model, row)

In [ ]:
vis_weight(model, row.layer_name, row.channel, 10)